# 03 — Thí nghiệm model và hybrid ensemble

Notebook so sánh baseline nghiệp vụ/thống kê, linear regression có regularization, hai tree ensemble và RNN thực sự với chuỗi 20 phiên. Model truyền thống dùng dòng feature tại `t`; RNN nhận một chuỗi theo thời gian. Hybrid ensemble kết hợp các component đã fit để giảm phụ thuộc vào một họ model. Việc chọn model dựa trên validation MAE và expanding-window CV. Holdout chỉ được đánh giá một lần sau khi khóa model chiến thắng. VIC chỉ là ticker ví dụ được cấu hình cho lần chạy này.

In [1]:
import json
import os
import warnings
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
MPL_CONFIG = ROOT / ".matplotlib"
MPL_CONFIG.mkdir(exist_ok=True)
os.environ["MPLCONFIGDIR"] = str(MPL_CONFIG)

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
# Run minh hoạ mặc định dùng VIC; pipeline nhận ticker khác qua config/CLI.
DATA_PATH = ROOT / "data" / "raw" / "VIC_VN.csv"
REPORT_DIR = ROOT / "evaluation"
REPORT_DIR.mkdir(parents=True, exist_ok=True)
SEED = 42
np.random.seed(SEED)


FEATURES = [
    "return_1d", "return_5d", "volatility_10", "volatility_20",
    "volume_z_20", "rsi_14", "macd_scaled", "price_ma20", "price_ma50",
]
TARGET = "target_return_5d"

def build_frame(raw: pd.DataFrame) -> pd.DataFrame:
    df = raw.copy().sort_values("date").drop_duplicates("date").reset_index(drop=True)
    close = pd.to_numeric(df["close"], errors="coerce")
    volume = pd.to_numeric(df["volume"], errors="coerce")
    returns = np.log(close / close.shift(1))
    delta = close.diff()
    gain = delta.clip(lower=0).ewm(alpha=1 / 14, adjust=False).mean()
    loss = (-delta.clip(upper=0)).ewm(alpha=1 / 14, adjust=False).mean()
    ema12 = close.ewm(span=12, adjust=False).mean()
    ema26 = close.ewm(span=26, adjust=False).mean()
    macd = ema12 - ema26
    df["return_1d"] = returns
    df["return_5d"] = np.log(close / close.shift(5))
    df["volatility_10"] = returns.rolling(10).std()
    df["volatility_20"] = returns.rolling(20).std()
    df["volume_z_20"] = (volume - volume.rolling(20).mean()) / volume.rolling(20).std()
    df["rsi_14"] = (100 - 100 / (1 + gain / loss.replace(0, np.nan))) / 100
    df["macd_scaled"] = macd / close
    df["price_ma20"] = close / close.rolling(20).mean() - 1
    df["price_ma50"] = close / close.rolling(50).mean() - 1
    df[TARGET] = np.log(close.shift(-5) / close)
    return df.replace([np.inf, -np.inf], np.nan)

In [2]:
from copy import deepcopy
from sklearn.base import clone
from sklearn.ensemble import ExtraTreesRegressor, HistGradientBoostingRegressor
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import TimeSeriesSplit
import torch
from torch import nn

torch.manual_seed(SEED)
torch.set_num_threads(1)

raw = pd.read_csv(DATA_PATH, parse_dates=["date"])
frame = build_frame(raw).dropna(subset=FEATURES + [TARGET]).reset_index(drop=True)
n = len(frame)
train_end, validation_end = int(n * .60), int(n * .80)
train, validation, holdout = frame.iloc[:train_end], frame.iloc[train_end:validation_end], frame.iloc[validation_end:]

models = {
    "ridge": Pipeline([("scale", StandardScaler()), ("model", Ridge(alpha=10.0))]),
    "extra_trees": ExtraTreesRegressor(n_estimators=250, min_samples_leaf=8, max_features=.8, random_state=SEED, n_jobs=1),
    "hist_gradient_boosting": HistGradientBoostingRegressor(max_iter=200, max_leaf_nodes=15, learning_rate=.04, l2_regularization=1.0, random_state=SEED),
}

def metrics(y, pred):
    return {
        "mae": float(mean_absolute_error(y, pred)),
        "rmse": float(mean_squared_error(y, pred) ** .5),
        "directional_accuracy": float(np.mean(np.sign(y) == np.sign(pred))),
    }

In [3]:
cv = TimeSeriesSplit(n_splits=4, gap=5)
cv_rows = []
combined = pd.concat([train, validation]).reset_index(drop=True)
for name, estimator in models.items():
    fold_mae = []
    for fold, (fit_idx, score_idx) in enumerate(cv.split(combined), 1):
        fitted = clone(estimator).fit(combined.loc[fit_idx, FEATURES], combined.loc[fit_idx, TARGET])
        pred = fitted.predict(combined.loc[score_idx, FEATURES])
        fold_mae.append(mean_absolute_error(combined.loc[score_idx, TARGET], pred))
    cv_rows.append({"model": name, "cv_mae_mean": np.mean(fold_mae), "cv_mae_std": np.std(fold_mae)})
cv_results = pd.DataFrame(cv_rows).sort_values("cv_mae_mean")
cv_results

  File "C:\Users\lqb46\Documents\Projects\StocKast\.venv\Lib\site-packages\joblib\externals\loky\backend\context.py", line 247, in _count_physical_cores
    cpu_count_physical = _count_physical_cores_win32()
                         ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\lqb46\Documents\Projects\StocKast\.venv\Lib\site-packages\joblib\externals\loky\backend\context.py", line 299, in _count_physical_cores_win32
    cpu_info = subprocess.run(
               ^^^^^^^^^^^^^^^
  File "C:\Users\lqb46\.cache\codex-runtimes\codex-primary-runtime\dependencies\python\Lib\subprocess.py", line 548, in run
    with Popen(*popenargs, **kwargs) as process:
         ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\lqb46\.cache\codex-runtimes\codex-primary-runtime\dependencies\python\Lib\subprocess.py", line 1026, in __init__
    self._execute_child(args, executable, preexec_fn, close_fds,
  File "C:\Users\lqb46\.cache\codex-runtimes\codex-primary-runtime\dependencies\python\Lib\subprocess.py", line

,model,cv_mae_mean,cv_mae_std
0,ridge,0.030571,0.002208
1,extra_trees,0.031908,0.001885
2,hist_gradient_boosting,0.036105,0.003359


In [4]:
validation_predictions = {
    "zero_return": np.zeros(len(validation)),
    "recent_5d_return": validation["return_5d"].to_numpy(),
}
fitted_models = {}
for name, estimator in models.items():
    fitted_models[name] = clone(estimator).fit(train[FEATURES], train[TARGET])
    validation_predictions[name] = fitted_models[name].predict(validation[FEATURES])

validation_rows = [
    {"model": name, **metrics(validation[TARGET], pred)}
    for name, pred in validation_predictions.items()
]
pd.DataFrame(validation_rows).sort_values("mae")

,model,mae,rmse,directional_accuracy
0,zero_return,0.029993,0.045045,0.015982
2,ridge,0.030924,0.046047,0.511416
3,extra_trees,0.031184,0.045825,0.470320
4,hist_gradient_boosting,0.033744,0.048987,0.484018
1,recent_5d_return,0.040649,0.059464,0.527397


In [5]:
SEQ_LEN = 20

def sequence_split(frame, start, end, scaler):
    values = scaler.transform(frame[FEATURES]).astype("float32")
    targets = frame[TARGET].to_numpy(dtype="float32")
    xs, ys, positions = [], [], []
    for pos in range(max(SEQ_LEN - 1, start), end):
        xs.append(values[pos - SEQ_LEN + 1: pos + 1])
        ys.append(targets[pos])
        positions.append(pos)
    return np.stack(xs), np.asarray(ys), np.asarray(positions)

class ReturnRNN(nn.Module):
    def __init__(self, inputs):
        super().__init__()
        self.rnn = nn.GRU(inputs, hidden_size=24, num_layers=1, batch_first=True)
        self.head = nn.Sequential(nn.Dropout(.15), nn.Linear(24, 1))
    def forward(self, x):
        output, _ = self.rnn(x)
        return self.head(output[:, -1]).squeeze(1)

def train_rnn(frame, train_end, validation_end, epochs=70):
    scaler = StandardScaler().fit(frame.iloc[:train_end][FEATURES])
    x_train, y_train, _ = sequence_split(frame, 0, train_end, scaler)
    x_val, y_val, val_pos = sequence_split(frame, train_end, validation_end, scaler)
    net = ReturnRNN(len(FEATURES))
    optimizer = torch.optim.Adam(net.parameters(), lr=2e-3, weight_decay=1e-4)
    loss_fn = nn.L1Loss()
    best, patience = None, 10
    for epoch in range(epochs):
        net.train(); order = torch.randperm(len(x_train))
        for batch in order.split(64):
            xb, yb = torch.from_numpy(x_train[batch]), torch.from_numpy(y_train[batch])
            optimizer.zero_grad(); loss = loss_fn(net(xb), yb); loss.backward(); optimizer.step()
        net.eval()
        with torch.no_grad(): val_loss = loss_fn(net(torch.from_numpy(x_val)), torch.from_numpy(y_val)).item()
        if best is None or val_loss < best[0] - 1e-6:
            best = (val_loss, deepcopy(net.state_dict()), epoch + 1); patience = 10
        else:
            patience -= 1
            if patience == 0: break
    net.load_state_dict(best[1]); net.eval()
    with torch.no_grad(): val_pred = net(torch.from_numpy(x_val)).numpy()
    return net, scaler, val_pred, val_pos, best[2]

rnn, rnn_scaler, rnn_val_pred, rnn_val_pos, best_epoch = train_rnn(frame, train_end, validation_end)
validation_predictions["gru_rnn"] = rnn_val_pred
rnn_validation_metrics = metrics(frame.loc[rnn_val_pos, TARGET], rnn_val_pred)
ensemble_components = ["ridge", "extra_trees", "hist_gradient_boosting", "gru_rnn"]
validation_predictions["hybrid_ensemble"] = np.mean(
    [validation_predictions[name] for name in ensemble_components], axis=0
)
ensemble_validation_metrics = metrics(
    frame.loc[rnn_val_pos, TARGET], validation_predictions["hybrid_ensemble"]
)
rnn_validation_metrics, ensemble_validation_metrics, best_epoch

({'mae': 0.029957672766628505,
  'rmse': 0.044905067749966714,
  'directional_accuracy': 0.4885844748858447},
 {'mae': 0.030709135575569233,
  'rmse': 0.045627985050321104,
  'directional_accuracy': 0.4908675799086758},
 10)

In [6]:
selection = pd.DataFrame(
    validation_rows
    + [
        {"model": "gru_rnn", **rnn_validation_metrics},
        {"model": "hybrid_ensemble", **ensemble_validation_metrics},
    ]
)
selection = selection.merge(cv_results, on="model", how="left")
selection["eligible"] = selection["model"].isin(
    ["ridge", "extra_trees", "hist_gradient_boosting", "gru_rnn", "hybrid_ensemble"]
)
winner = selection[selection.eligible].sort_values(["mae", "cv_mae_std"], na_position="last").iloc[0]["model"]
print(selection.sort_values("mae").to_string(index=False))
print("MODEL ĐÃ KHÓA TRƯỚC KHI MỞ HOLDOUT:", winner)

                 model      mae     rmse  directional_accuracy  cv_mae_mean  cv_mae_std  eligible
               gru_rnn 0.029958 0.044905              0.488584          NaN         NaN      True
           zero_return 0.029993 0.045045              0.015982          NaN         NaN     False
       hybrid_ensemble 0.030709 0.045628              0.490868          NaN         NaN      True
                 ridge 0.030924 0.046047              0.511416     0.030571    0.002208      True
           extra_trees 0.031184 0.045825              0.470320     0.031908    0.001885      True
hist_gradient_boosting 0.033744 0.048987              0.484018     0.036105    0.003359      True
      recent_5d_return 0.040649 0.059464              0.527397          NaN         NaN     False
MODEL ĐÃ KHÓA TRƯỚC KHI MỞ HOLDOUT: gru_rnn


In [7]:
# Holdout được dùng lần đầu và duy nhất sau khi `winner` đã được khóa ở trên.
if winner in {"gru_rnn", "hybrid_ensemble"}:
    final_rnn, final_scaler, _, _, _ = train_rnn(frame, validation_end, validation_end + 1, epochs=max(25, best_epoch + 5))
    x_hold, y_hold, hold_pos = sequence_split(frame, validation_end, len(frame), final_scaler)
    with torch.no_grad(): hold_pred = final_rnn(torch.from_numpy(x_hold)).numpy()
    hold_dates = frame.loc[hold_pos, "date"].to_numpy()
    if winner == "hybrid_ensemble":
        final_traditional = {}
        for name, estimator in models.items():
            final_traditional[name] = clone(estimator).fit(
                frame.iloc[:validation_end][FEATURES], frame.iloc[:validation_end][TARGET]
            )
        traditional_hold = np.column_stack(
            [final_traditional[name].predict(frame.loc[hold_pos, FEATURES]) for name in models]
        )
        hold_pred = np.mean(np.column_stack([traditional_hold, hold_pred]), axis=1)
else:
    final_model = clone(models[winner]).fit(frame.iloc[:validation_end][FEATURES], frame.iloc[:validation_end][TARGET])
    hold_pred = final_model.predict(holdout[FEATURES])
    y_hold = holdout[TARGET].to_numpy()
    hold_dates = holdout["date"].to_numpy()

holdout_metrics = metrics(y_hold, hold_pred)
experiment = {
    "target": TARGET,
    "horizon_sessions": 5,
    "selected_model": winner,
    "selection_rule": "validation MAE thấp nhất trong các model có thể train; độ ổn định CV dùng để phân xử khi bằng nhau",
    "validation": selection.fillna(value={"cv_mae_mean": -1, "cv_mae_std": -1}).to_dict("records"),
    "holdout": holdout_metrics,
    "holdout_rows": int(len(y_hold)),
    "holdout_start": str(pd.Timestamp(hold_dates[0]).date()),
    "holdout_end": str(pd.Timestamp(hold_dates[-1]).date()),
    "rnn_sequence_length": SEQ_LEN,
    "rnn_best_epoch": int(best_epoch),
}
(REPORT_DIR / "model_metrics.json").write_text(json.dumps(experiment, indent=2), encoding="utf-8")
pd.DataFrame({"date": hold_dates, "actual": y_hold, "prediction": hold_pred}).to_csv(REPORT_DIR / "holdout_predictions.csv", index=False)
print(json.dumps(experiment, indent=2))

{
  "target": "target_return_5d",
  "horizon_sessions": 5,
  "selected_model": "gru_rnn",
  "selection_rule": "validation MAE th\u1ea5p nh\u1ea5t trong c\u00e1c model c\u00f3 th\u1ec3 train; \u0111\u1ed9 \u1ed5n \u0111\u1ecbnh CV d\u00f9ng \u0111\u1ec3 ph\u00e2n x\u1eed khi b\u1eb1ng nhau",
  "validation": [
    {
      "model": "zero_return",
      "mae": 0.029992757578356455,
      "rmse": 0.04504478780898461,
      "directional_accuracy": 0.01598173515981735,
      "cv_mae_mean": -1.0,
      "cv_mae_std": -1.0,
      "eligible": false
    },
    {
      "model": "recent_5d_return",
      "mae": 0.04064892263807127,
      "rmse": 0.059463974151034084,
      "directional_accuracy": 0.5273972602739726,
      "cv_mae_mean": -1.0,
      "cv_mae_std": -1.0,
      "eligible": false
    },
    {
      "model": "ridge",
      "mae": 0.030924222286903082,
      "rmse": 0.04604699007137938,
      "directional_accuracy": 0.5114155251141552,
      "cv_mae_mean": 0.03057061536376185,
      "cv_ma